# Features and the Parsons gap

The original analysis already had quality features (first / best score by
activity family). The DuckDB layer adds coverage, timing, and mix. This notebook
reads the merged table so we aren't rescanning 20M rows.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))

from ebook_analysis.stats import add_cohort, compute_corr_table, within_group_quartile_gap

out = ROOT / "analysis_outputs"
features = pd.read_parquet(out / "student_exam_features_sql.parquet")
corr = pd.read_csv(out / "activity_correlations_with_sql.csv")
features.shape


Quick look at the new columns. Anything starting with `coverage__`, `timing__`,
`consistency__`, or `mix__` came from SQL.


In [ ]:
new_cols = [c for c in features.columns if c.startswith(("coverage__", "timing__", "consistency__", "mix__"))]
new_cols
features[new_cols].describe().T.head(20)


The number everyone quotes is the within-cohort quartile gap for Parsons
first-try quality. "Within cohort" just means: rank students against other
people taking the same midterm in the same semester. Otherwise F21 mid1 and
W24 mid2 get dumped into one comparison, which is messy.


In [ ]:
parsons = corr.loc[corr["feature"].eq("parsons__mean_first_score")].iloc[0]
parsons


In [ ]:
gap = within_group_quartile_gap(
    add_cohort(features), "cohort", "parsons__mean_first_score", "score_pct"
)
print(f"top vs bottom Parsons quartile: {gap:.1f} points")


Do the new coverage / timing features even matter? Quality still wins, but
chapter coverage is a nicer "did they actually touch the topics?" check than
raw event counts.


In [ ]:
focus = [
    "parsons__mean_first_score",
    "mchoice__mean_first_score",
    "coverage__unique_chapters",
    "coverage__chapter_diversity",
    "timing__share_last_7d",
    "consistency__span_days",
    "mix__coding_share",
    "all_activity__total_events",
]
corr.loc[corr["feature"].isin(focus)].sort_values("within_group_rank_corr", ascending=False)


One thing that looks backwards until you sit with it: `mix__parsons_share` is
*negatively* related to exam score, while Parsons first-try quality is the
strongest positive feature. Students who spend a huge fraction of their
clicks on Parsons but don't get them right on the first check are exactly
the high-effort / low-efficiency group. Volume without quality is not a
good sign.


In [ ]:
corr.loc[corr["feature"].isin(["parsons__mean_first_score", "mix__parsons_share", "mix__activecode_share"])]


In [ ]:
quartiles = pd.read_csv(out / "feature_quartile_scores.csv")
quartiles.loc[quartiles["feature"].eq("parsons__mean_first_score")].groupby("feature_quartile", observed=False)["mean_score_pct"].mean()
